In [1]:
# 1. Install required dependencies for embeddings, vector storage, PDF parsing, BM25, and Gemini API
!pip install -q sentence-transformers chromadb pymupdf rank_bm25 tqdm google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
# 2. Mount Google Drive for persistent file access and artifact storage
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 3. Load Gemini API Key from Google Colab Secrets
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
print("Gemini API key loaded.")

✓ Gemini API key φορτώθηκε.


In [4]:
# 4. Verify input PDF path existence on Google Drive
import os

pdf_filename = "/content/drive/MyDrive/Thesis/762-ΠΑΠΑΔΟΠΟΥΛΟΣ-Technologies-for-Big-Data-Analytics.pdf"

if os.path.exists(pdf_filename):
    print(f"File found: {pdf_filename}")
else:
    print(f"Warning: File NOT found at path: {pdf_filename}")
    print("Check your file name and path in Google Drive.")

✓ Το αρχείο βρέθηκε επιτυχώς: /content/drive/MyDrive/Thesis/762-ΠΑΠΑΔΟΠΟΥΛΟΣ-Technologies-for-Big-Data-Analytics.pdf


In [5]:
# 5. Global configuration parameters, model selection, and GPU availability check
import torch

# Dense vector embedding and multimodal vision model configurations
EMBEDDING_MODEL_NAME  = 'intfloat/multilingual-e5-large-instruct'
GEMINI_VISION_MODEL   = 'gemini-2.5-flash'

# Sliding window text chunking hyperparameters
CHUNK_SIZE            = 400
CHUNK_OVERLAP         = 100
# Minimum word count threshold to discard meaningless header/footer text fragments
MIN_CHUNK_WORDS       = 30

# Vector database and sparse BM25 index storage paths
CHROMA_PATH           = "chroma_db"
COLLECTION_NAME       = "bigdata_book"
BM25_INDEX            = "bm25_bigdata.pkl"

# Image dimensions and area thresholds to filter out decorative icons and tiny logos
MIN_IMAGE_WIDTH       = 150
MIN_IMAGE_HEIGHT      = 150
MIN_IMAGE_AREA        = 50_000

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found! Go to Runtime Change runtime type T4 GPU")

GPU διαθέσιμη: True
GPU: Tesla T4


In [6]:
# 6. Text processing, chapter heading detection, and sliding window chunking functions
import re
import string
from typing import List, Dict, Any, Optional

def tokenize_greek(text: str) -> List[str]:
    """Cleans and tokenizes Greek text into terms for BM25 indexing.

    Args:
        text: Raw text string to process.

    Returns:
        List of cleaned, tokenized strings with punctuation removed.
    """
    text = text.lower().replace('-', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    return [t for t in text.split() if len(t) > 1]

def detect_chapter(line: str) -> Optional[str]:
    """Detects chapter and section headings from PDF text stream using structural heuristics.

    Args:
        line: Single line of text extracted from a PDF page.

    Returns:
        Detected heading string truncated to 80 characters, or None if line is not a title.
    """
    line = line.strip()
    if not line:
        return None
    # Skip isolated page numbers
    if re.fullmatch(r'\d+', line):
        return None
    # Match explicit chapter headings (e.g. "Chapter 1" or "Κεφάλαιο 1")
    if re.match(r'^(κεφάλαιο|chapter)\s+\d+', line, re.IGNORECASE):
        return line[:80]
    # Match numbered section headings (e.g. "1.2 Section Title") containing 2 to 12 words
    m = re.match(r'^(\d+)(\.\d+){0,2}\s+(.+)$', line)
    if m and 2 <= len(line.split()) <= 12:
        return line[:80]
    # Heuristic: Short uppercase lines with at least 4 letters usually denote section titles
    letters = [c for c in line if c.isalpha()]
    if letters and len(letters) >= 4 and all(c.isupper() for c in letters) and len(line) <= 80:
        return line
    return None

def sliding_window_chunks(words: List[str], page_map: List[int], chapter_map: List[Optional[str]]) -> List[Dict[str, Any]]:
    """Splits word stream into overlapping chunks while tracking word-level page and chapter boundaries.

    Args:
        words: Sequential list of words extracted from document.
        page_map: List mapping each word index to its 1-indexed page number.
        chapter_map: List mapping each word index to its active chapter heading.

    Returns:
        List of dictionaries containing chunk text, start page, end page, and chapter metadata.
    """
    chunks, start, total = [], 0, len(words)
    while start < total:
        end         = min(start + CHUNK_SIZE, total)
        chunk_words = words[start:end]
        if len(chunk_words) >= MIN_CHUNK_WORDS:
            chunks.append({
                "text":       " ".join(chunk_words),
                "page_start": page_map[start],
                "page_end":   page_map[end - 1],
                "chapter":    chapter_map[start] or "-",
            })
        if end == total:
            break
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks

print("Helper functions loaded.")

✓ Βοηθητικές συναρτήσεις φορτώθηκαν.


In [7]:
# 7. Image description extraction using Gemini Vision API with persistent caching and rate-limit backoff
import hashlib
import time
import json
import os
from typing import Tuple
from google import genai
from google.genai import types

vision_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
# Persistent cache file path on Google Drive to prevent duplicate API calls across execution runs
CACHE_FILE = "/content/drive/MyDrive/Thesis/vision_cache.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        vision_cache = json.load(f)
    print(f"Loaded {len(vision_cache)} cached image descriptions.")
else:
    vision_cache = {}
    print("Created new empty vision cache.")

def save_cache() -> None:
    """Persists the vision description cache dictionary to disk."""
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(vision_cache, f, ensure_ascii=False, indent=2)

def describe_image(image_bytes: bytes, image_ext: str, page_num: int, surrounding_text: str) -> Tuple[str, bool]:
    """Generates textual description for an image using Gemini Vision API with persistent caching and exponential retry backoff.

    Args:
        image_bytes: Raw binary content of the image.
        image_ext: Image format extension (e.g. png, jpg).
        page_num: Page number where the image was extracted.
        surrounding_text: Text extracted from the parent page to provide context.

    Returns:
        Tuple of (description_string, is_cached_boolean).
    """
    # Hash raw image payload using MD5 to check persistent cache for exact image deduplication
    img_hash = hashlib.md5(image_bytes).hexdigest()

    if img_hash in vision_cache:
        return vision_cache[img_hash], True

    # Truncate surrounding page text to first 100 words to ground the vision model without overloading context
    context_hint = " ".join(surrounding_text.split()[:100]) if surrounding_text else ""
    prompt = "Περίγραψε αυτή την εικόνα περιεκτικά στα Ελληνικά. "
    if context_hint:
        prompt += f"Το κείμενο της σελίδας γύρω από την εικόνα είναι: {context_hint}. "
    prompt += "Αν είναι διάγραμμα: γράψε τα components και τη ροή. Αν είναι πίνακας: κάνε transcribe τα δεδομένα. Αν είναι κώδικας: αντέγραψέ τον ακριβώς."

    mime_type = "image/jpeg" if image_ext.lower() in ("jpg", "jpeg") else f"image/{image_ext.lower()}"

    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = vision_client.models.generate_content(
                model=GEMINI_VISION_MODEL,
                contents=[
                    prompt,
                    types.Part.from_bytes(data=image_bytes, mime_type=mime_type)
                ]
            )
            result = response.text.strip()

            vision_cache[img_hash] = result
            save_cache()

            return result, False

        except Exception as e:
            error_msg = str(e)
            # Handle rate limit status codes (429/503) by waiting 65s for window reset
            if "429" in error_msg or "503" in error_msg:
                wait_time = 65
                print(f"
  [API Limit] Page {page_num}: Waiting {wait_time}s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"
  Vision Error (Page {page_num}): {e}")
                return "", False

    print(f"
  Failed image description after {max_retries} attempts.")
    return "", False

print("Vision function ready.")

✓ Φορτώθηκαν 16 αποθηκευμένες περιγραφές από το cache.
✓ Vision συνάρτηση έτοιμη.


In [8]:
# 8. Document processing: text extraction and image parsing with Gemini Vision descriptions
import fitz
from tqdm import tqdm

doc         = fitz.open(pdf_filename)
total_pages = len(doc)
print(f"Pages: {total_pages}")

all_words, page_map, chapter_map = [], [], []
current_chapter = None
page_texts      = {}

print("
Extracting text...")
for page_num in tqdm(range(total_pages)):
    page = doc.load_page(page_num)
    raw  = page.get_text("text").strip()
    page_texts[page_num] = raw
    for line in raw.splitlines():
        ch = detect_chapter(line)
        if ch:
            current_chapter = ch
        for w in line.split():
            all_words.append(w)
            page_map.append(page_num + 1)
            chapter_map.append(current_chapter)

print(f"{len(all_words)} words extracted.")

print("
Extracting and describing images with Gemini...")

image_chunks = []
image_count  = 0

for page_num in tqdm(range(total_pages), desc="Pages"):
    page       = doc.load_page(page_num)
    image_list = page.get_images(full=True)

    for img_idx, img in enumerate(image_list):
        try:
            xref       = img[0]
            image_data = doc.extract_image(xref)
            w = image_data.get("width",  0)
            h = image_data.get("height", 0)

            # Filter out small logos, icons, and low-resolution graphics
            if w < MIN_IMAGE_WIDTH or h < MIN_IMAGE_HEIGHT:
                continue
            if w * h < MIN_IMAGE_AREA:
                continue

            image_bytes = image_data["image"]
            image_ext   = image_data.get("ext", "png")
            surrounding = page_texts.get(page_num, "")

            print(f"  Page {page_num+1} Img {img_idx+1} ({w}x{h}px)...", end=" ")

            vision_text, is_cached = describe_image(image_bytes, image_ext, page_num + 1, surrounding)

            if len(vision_text.strip()) < 20:
                print("skipped (empty description)")
                continue

            if is_cached:
                print(f"[CACHED] ({len(vision_text)} chars)")
            else:
                print(f"({len(vision_text)} chars)")
                # Rate-limit delay (10s) enforced only for fresh API calls
                time.sleep(10)

            image_chunks.append({
                "text":       f"[ΕΙΚΟΝΑ - Σελίδα {page_num+1}] Περιγραφή: {vision_text}",
                "source":     pdf_filename,
                "type":       "image",
                "page_start": page_num + 1,
                "page_end":   page_num + 1,
                "chapter":    current_chapter or "-",
            })
            image_count += 1

        except Exception as e:
            print(f"  Error page {page_num} img {img_idx}: {e}")
            continue

print(f"
{image_count} images described (API or cache).")
doc.close()

Σελίδες: 362

Πέρασμα 1: Εξαγωγή κειμένου...


100%|██████████| 362/362 [00:01<00:00, 192.79it/s]


✓ 133285 λέξεις εξήχθησαν.

Πέρασμα 2: Εξαγωγή και περιγραφή εικόνων με Gemini...


Σελίδες:   0%|          | 0/362 [00:00<?, ?it/s]

  🖼 Σελ.1 Εικ.1 (1868×2803px)... ✓ [CACHED] (839 χαρ.)
  🖼 Σελ.1 Εικ.2 (317×323px)... ✓ (556 χαρ.)


...

  🖼 Σελ.268 Εικ.1 (481×511px)... ✓ (1323 χαρ.)


Σελίδες:  74%|███████▍  | 268/362 [03:54<02:36,  1.66s/it]

  🖼 Σελ.285 Εικ.1 (1000×352px)... ✓ [CACHED] (1039 χαρ.)
  🖼 Σελ.298 Εικ.1 (313×304px)... ✓ (646 χαρ.)


Σελίδες: 100%|██████████| 362/362 [04:12<00:00,  1.43it/s]

  🖼 Σελ.329 Εικ.1 (1000×352px)... ✓ [CACHED] (1039 χαρ.)

✓ 40 εικόνες περιγράφηκαν (είτε από το API είτε από το cache).


In [9]:
# 9. Aggregate text chunks and vision image descriptions into unified chunk list
all_chunks = []

print("Creating text chunks...")
text_chunks = sliding_window_chunks(all_words, page_map, chapter_map)
for ch in text_chunks:
    all_chunks.append({
        "text":       ch["text"],
        "source":     pdf_filename,
        "type":       "text",
        "page_start": ch["page_start"],
        "page_end":   ch["page_end"],
        "chapter":    ch["chapter"],
    })
print(f"{len(text_chunks)} text chunks")

all_chunks.extend(image_chunks)
print(f"{len(image_chunks)} image chunks")
print(f"
Total chunks: {len(all_chunks)}")

Δημιουργία text chunks...
→ 444 text chunks
→ 40 image chunks

Σύνολο chunks: 484


In [10]:
# 10. Generate GPU embeddings using multilingual E5 model
from sentence_transformers import SentenceTransformer

print(f"Loading model: {EMBEDDING_MODEL_NAME}")
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = [c["text"] for c in all_chunks]
print(f"Generating embeddings for {len(texts)} chunks...")
embeddings = embed_model.encode(texts, show_progress_bar=True, batch_size=64).tolist()
print(f"Embeddings: {len(embeddings)} x {len(embeddings[0])} dims")

Φόρτωση μοντέλου: intfloat/multilingual-e5-large-instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Δημιουργία embeddings για 484 chunks...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

✓ Embeddings: 484 x 1024 dims


In [11]:
# 11. Populate ChromaDB vector store and build BM25 sparse index
import chromadb
import pickle
from rank_bm25 import BM25Okapi

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

ids = [f"chunk_{i}" for i in range(len(all_chunks))]
chroma_metadata = [
    {
        "source":     c["source"],
        "type":       c["type"],
        "page_start": int(c["page_start"]),
        "page_end":   int(c["page_end"]),
        "chapter":    str(c["chapter"]),
    }
    for c in all_chunks
]

collection.add(documents=texts, embeddings=embeddings, metadatas=chroma_metadata, ids=ids)
print(f"{len(all_chunks)} chunks saved to ChromaDB.")

print("Building BM25 index...")
bm25 = BM25Okapi([tokenize_greek(t) for t in texts])
with open(BM25_INDEX, "wb") as f:
    pickle.dump({"bm25": bm25, "chunks": texts, "metadata": chroma_metadata, "ids": ids}, f)
print("BM25 index saved.")

✓ 484 chunks αποθηκεύτηκαν στη ChromaDB.
Χτίσιμο BM25 index...
✓ BM25 index αποθηκεύτηκε.


In [12]:
# 12. Save image description log, compress vector database, and trigger automatic downloads
import shutil
import json
from google.colab import files

image_descriptions = [
    {"page": c["page_start"], "description": c["text"]}
    for c in all_chunks if c["type"] == "image"
]
with open("image_descriptions_greek.json", "w", encoding="utf-8") as f:
    json.dump(image_descriptions, f, ensure_ascii=False, indent=2)
print(f"{len(image_descriptions)} descriptions saved to image_descriptions_greek.json")

shutil.make_archive("chroma_db", "zip", "chroma_db")

print("
Downloading files...")
files.download("chroma_db.zip")
files.download(BM25_INDEX)
files.download("image_descriptions_greek.json")

print("
Finished!")

✓ 40 περιγραφές αποθηκεύτηκαν στο image_descriptions_greek.json

Κατέβασμα αρχείων...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Ολοκληρώθηκε!
Οδηγίες:
1. Αποσύμπιεσε το chroma_db.zip στο φάκελο rag-procect-book/
2. Βάλε το bm25_bigdata.pkl στο φάκελο rag-procect-book/
3. Τρέξε: python main.py
